# Gaussian Process

This demo shows how to use GaussianProcessRegressor to regress the MET efficiency (MET<50) for three-jet events, with respect to the jet-energy scales.

Four nuisance parameters are considered:  
1) J_central: jet scale of the leading jet with |eta| < 1  
2) J_forward: jet scale of the leading jet with |eta| >= 1  
3) j_central: jet scale of the two subleading jets with |eta| < 1  
4) j_forward: jet scale of the two subleading jets with |eta| >= 1  

In [1]:
import sys 
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error

sys.path.append("../..")
import gpder
from gpder.gaussian_process import GaussianProcessRegressor
from gpder.gaussian_process.kernels import GPKernel, GPKernelDerAware

try:
    import uproot
except:
    print("uproot must be installed to run this demo.")

The three-momenta of the jets are stored in the file "three_jets.root". The jets are sorted in decreasing pT scale.

Load the jets and keep only those with events with three jets.

In [2]:
f = uproot.open("./three_jets.root")
tree = f['tnt']

njets = np.array(tree['nj'])
njets_cut = np.where(njets == 3)

j1_threeM = np.stack((np.array(tree['j1_pt'])[njets_cut],
                      np.array(tree['j1_eta'])[njets_cut],
                      np.array(tree['j1_phi'])[njets_cut]), axis=1)

j2_threeM = np.stack((np.array(tree['j2_pt'])[njets_cut],
                      np.array(tree['j2_eta'])[njets_cut],
                      np.array(tree['j2_phi'])[njets_cut]), axis=1)

j3_threeM = np.stack((np.array(tree['j3_pt'])[njets_cut],
                      np.array(tree['j3_eta'])[njets_cut],
                      np.array(tree['j3_phi'])[njets_cut]), axis=1)

f.close()

Select the training and testing points. 

The training points are a grid small grid at [0.9, 1.0, 1.1] along each NP dimension.  
The testing points are a grid of 10-points spanning the [0.7, 0.3] range in each NP dimension.

In [3]:
# -- training points -- # 
train_pts = [0.9, 1.0, 1.1]

X_train = np.array([[Jc, Jf, jc, jf] for Jc in train_pts \
                                     for Jf in train_pts \
                                     for jc in train_pts \
                                     for jf in train_pts])


# -- testing points -- # 
res=5
X_lower = 0.7
X_upper = 1.3
test_pts = np.linspace(X_lower, X_upper, res)
X_test = np.array([[Jc, Jf, jc, jf] for Jc in test_pts \
                                    for Jf in test_pts \
                                    for jc in test_pts \
                                    for jf in test_pts])

## Regular GPR

The function Eff_MET50_sigmoid below calculates the efficiency of the events with MET<50.

In [4]:
def sigmoid(X):
    return 1. / (1. + np.exp(-X))

In [5]:
def Eff_MET50(J_central, J_forward, 
              j_central, j_forward, 
              j1_threeM=j1_threeM, 
              j2_threeM=j2_threeM, 
              j3_threeM=j3_threeM):
    count_met = 0
    count_pTcut = 0
    for i in range(len(j1_threeM)):
        # -- scale the pT of the leading subjet -- # 
        if abs(j1_threeM[i, 1]) < 1:
            j1_pt = j1_threeM[i, 0] / J_central
        else:
            j1_pt = j1_threeM[i, 0] / J_forward
        # -- and of the two subleading subjets -- #
        if abs(j2_threeM[i, 1]) < 1:
            j2_pt = j2_threeM[i, 0] / j_central
            j3_pt = j3_threeM[i, 0] / j_central
        else:
            j2_pt = j2_threeM[i, 0] / j_forward
            j3_pt = j3_threeM[i, 0] / j_forward
            
        if (j1_pt > 200 and j2_pt  < 200):
            count_pTcut += 1
            # -- pT_x decomp -- # 
            j1_pt_x = j1_pt * np.cos(j1_threeM[i, 2])
            j2_pt_x = j2_pt * np.cos(j2_threeM[i, 2])
            j3_pt_x = j3_pt * np.cos(j3_threeM[i, 2])
            # -- pT_y decomp -- #
            j1_pt_y = j1_pt * np.sin(j1_threeM[i, 2])
            j2_pt_y = j2_pt * np.sin(j2_threeM[i, 2])
            j3_pt_y = j3_pt * np.sin(j3_threeM[i, 2])
            # -- MET -- #
            met_x = j1_pt_x + j2_pt_x + j3_pt_x
            met_y = j1_pt_y + j2_pt_y + j3_pt_y
            met = np.sqrt(met_x * met_x + met_y * met_y)

            if met < 50:
                count_met += 1

    return count_met / count_pTcut

In [6]:
def Eff_MET50_sigmoid(J_central, J_forward, 
                      j_central, j_forward, 
                      j1_threeM=j1_threeM, 
                      j2_threeM=j2_threeM, 
                      j3_threeM=j3_threeM):
    count_met = 0
    count_pTcut = 0
    for i in range(len(j1_threeM)):
        # -- scale the pT of the leading subjet -- # 
        if abs(j1_threeM[i, 1]) < 1:
            j1_pt = j1_threeM[i, 0] / J_central
        else:
            j1_pt = j1_threeM[i, 0] / J_forward
        # -- and of the two subleading subjets -- #
        if abs(j2_threeM[i, 1]) < 1:
            j2_pt = j2_threeM[i, 0] / j_central
            j3_pt = j3_threeM[i, 0] / j_central
        else:
            j2_pt = j2_threeM[i, 0] / j_forward
            j3_pt = j3_threeM[i, 0] / j_forward
            
        if (j1_pt > 200 and j2_pt  < 200):
            count_pTcut += 1
            # -- pT_x decomp -- # 
            j1_pt_x = j1_pt * np.cos(j1_threeM[i, 2])
            j2_pt_x = j2_pt * np.cos(j2_threeM[i, 2])
            j3_pt_x = j3_pt * np.cos(j3_threeM[i, 2])
            # -- pT_y decomp -- #
            j1_pt_y = j1_pt * np.sin(j1_threeM[i, 2])
            j2_pt_y = j2_pt * np.sin(j2_threeM[i, 2])
            j3_pt_y = j3_pt * np.sin(j3_threeM[i, 2])
            # -- MET -- #
            met_x = j1_pt_x + j2_pt_x + j3_pt_x
            met_y = j1_pt_y + j2_pt_y + j3_pt_y
            met = np.sqrt(met_x * met_x + met_y * met_y)

            count_met += sigmoid(-(met-50.0))

    return count_met / count_pTcut

In [7]:
y_train = np.array([Eff_MET50_sigmoid(X_train[i][0], 
                                      X_train[i][1],
                                      X_train[i][2],
                                      X_train[i][3]) for i in range(len(X_train))])

y_test = np.array([Eff_MET50_sigmoid(X_test[i][0], 
                                     X_test[i][1],
                                     X_test[i][2],
                                     X_test[i][3]) for i in range(len(X_test))])

In [8]:
kernel = GPKernel()
gp = GaussianProcessRegressor(kernel=kernel,
                              n_restarts_optimizer=10, 
                              random_state=123)
gp.fit(X=X_train, y=y_train)
mu, cov = gp.predict(X=X_test, return_cov=True)
mse = mean_squared_error(mu, y_test)
print(mse)

0.07338210962611162


## Derivative-enhanced GPR

In [9]:
def dsigmoid(X):
    return sigmoid(X) * (1 - sigmoid(X))

In [10]:
def dEff_MET50_sigmoid(J_central, J_forward, 
                      j_central, j_forward, 
                      j1_threeM=j1_threeM, 
                      j2_threeM=j2_threeM, 
                      j3_threeM=j3_threeM):
    
    count_met = 0
    count_pTcut = 0
    dmet_dJcentral = 0
    dmet_dJforward = 0
    dmet_djcentral = 0
    dmet_djforward = 0
    for i in range(len(j1_threeM)):
        # -- scale the pT of the leading subjet -- # 
        if abs(j1_threeM[i, 1]) < 1:
            j1_pt = j1_threeM[i, 0] / J_central
        else:
            j1_pt = j1_threeM[i, 0] / J_forward
        # -- and of the two subleading subjets -- #
        if abs(j2_threeM[i, 1]) < 1:
            j2_pt = j2_threeM[i, 0] / j_central
            j3_pt = j3_threeM[i, 0] / j_central
        else:
            j2_pt = j2_threeM[i, 0] / j_forward
            j3_pt = j3_threeM[i, 0] / j_forward
            
        if (j1_pt > 200 and j2_pt  < 200):
            count_pTcut += 1
            # -- pT_x decomp -- # 
            j1_pt_x = j1_pt * np.cos(j1_threeM[i, 2])
            j2_pt_x = j2_pt * np.cos(j2_threeM[i, 2])
            j3_pt_x = j3_pt * np.cos(j3_threeM[i, 2])
            # -- pT_y decomp -- #
            j1_pt_y = j1_pt * np.sin(j1_threeM[i, 2])
            j2_pt_y = j2_pt * np.sin(j2_threeM[i, 2])
            j3_pt_y = j3_pt * np.sin(j3_threeM[i, 2])            
            # -- MET -- #
            met_x = j1_pt_x + j2_pt_x + j3_pt_x
            met_y = j1_pt_y + j2_pt_y + j3_pt_y
            met = np.sqrt(met_x * met_x + met_y * met_y)
            
            # -- chain rule terms -- #
            dsig = dsigmoid(-(met-50.0)) * (-1)
            dmet = 0.5 / met 
            if abs(j1_threeM[i, 1]) < 1:
                dJ_central  = -2 * met_x * j1_threeM[i, 0] * np.cos(j1_threeM[i, 2]) / J_central**2 
                dJ_central += -2 * met_y * j1_threeM[i, 0] * np.sin(j1_threeM[i, 2]) / J_central**2 
                dJ_forward  = 0
            else:
                dJ_central  = 0
                dJ_forward  = -2 * met_x * j1_threeM[i, 0] * np.cos(j1_threeM[i, 2]) / J_forward**2 
                dJ_forward += -2 * met_y * j1_threeM[i, 0] * np.sin(j1_threeM[i, 2]) / J_forward**2 
            if abs(j2_threeM[i, 1]) < 1:
                dj_central  = -2 * met_x * j2_threeM[i, 0] * np.cos(j2_threeM[i, 2]) / j_central**2
                dj_central += -2 * met_x * j3_threeM[i, 0] * np.cos(j3_threeM[i, 2]) / j_central**2
                dj_central += -2 * met_y * j2_threeM[i, 0] * np.sin(j2_threeM[i, 2]) / j_central**2
                dj_central += -2 * met_y * j3_threeM[i, 0] * np.sin(j3_threeM[i, 2]) / j_central**2
                dj_forward  = 0
            else:
                dj_central  = 0
                dj_forward  = -2 * met_x * j2_threeM[i, 0] * np.cos(j2_threeM[i, 2]) / j_forward**2
                dj_forward += -2 * met_x * j3_threeM[i, 0] * np.cos(j3_threeM[i, 2]) / j_forward**2
                dj_forward += -2 * met_y * j2_threeM[i, 0] * np.sin(j2_threeM[i, 2]) / j_forward**2
                dj_forward += -2 * met_y * j3_threeM[i, 0] * np.sin(j3_threeM[i, 2]) / j_forward**2
            
            dmet_dJcentral += dsig * dmet * dJ_central
            dmet_dJforward += dsig * dmet * dJ_forward
            dmet_djcentral += dsig * dmet * dj_central
            dmet_djforward += dsig * dmet * dj_forward

    return [dmet_dJcentral / count_pTcut, dmet_dJforward / count_pTcut,
            dmet_djcentral / count_pTcut, dmet_djforward / count_pTcut]

In [11]:
dX_train = X_train
dy_train = np.array([dEff_MET50_sigmoid(dX_train[i][0], 
                                        dX_train[i][1],
                                        dX_train[i][2],
                                        dX_train[i][3]) for i in range(len(dX_train))])

In [12]:
kernel = GPKernelDerAware()
gp = GaussianProcessRegressor(kernel=kernel,
                              n_restarts_optimizer=10, 
                              random_state=123)
gp.fit(X=X_train, y=y_train,
       dX=dX_train, dy=dy_train)
mu, cov = gp.predict(X=X_test, return_cov=True)
mse = mean_squared_error(mu, y_test)
print(mse)

0.06241018801163702
